In [1]:
import torch.nn as nn
import torch 
from blocks import ExtractFbanks64
from blocks import ResNetModel
from blocks import StatPoolLayer
from blocks import MaxoutSegmentLevel
from blocks import CurricularAAM 
from model import VerificationModel

In [15]:
# пути до тестового протокола. подгружаются через wav.scp и файлы target и imposter-пар
scp_path = '/mnt/hs/dorado1/anikin/VoxCeleb1/protocols/VoxCeleb1-O/wav.scp'
protocol_path = '/mnt/hs/dorado1/anikin/VoxCeleb1/protocols/VoxCeleb1-O/'

### Загрузка данных для примера

In [16]:
from metrics.test import prepare_pandas_protocol
prepare_pandas_protocol(protocol_path)

,enroll,test,is_target
0,id10270/x6uYqmx31kE/00001.wav,id10300/ize_eiCFEg0/00003.wav,0
1,id10270/x6uYqmx31kE/00001.wav,id10273/0OCW1HUxZyg/00001.wav,0
2,id10270/x6uYqmx31kE/00001.wav,id10284/Uzxv7Axh3Z8/00001.wav,0
3,id10270/x6uYqmx31kE/00001.wav,id10284/7yx9A0yzLYk/00029.wav,0
4,id10270/x6uYqmx31kE/00002.wav,id10285/m-uILToQ9ss/00009.wav,0
...,...,...,...
18797,id10309/0cYFdtyWVds/00004.wav,id10309/tGEWD2GaiDw/00003.wav,1
18798,id10309/0cYFdtyWVds/00005.wav,id10309/rqaAm4bEsXc/00006.wav,1
18799,id10309/0cYFdtyWVds/00005.wav,id10309/nMZkxUQ1RqI/00001.wav,1
18800,id10309/0cYFdtyWVds/00005.wav,id10309/0b1inHMAr6o/00010.wav,1


### ResNet

In [4]:
# инициализируем компоненты модели
fe = ExtractFbanks64()
frame_level = ResNetModel.factory('channels_96')
pooling = StatPoolLayer.by_string_mode('MV')
segment = MaxoutSegmentLevel(input_dim=1536, output_dim=512,enable_batch_norm=True)

# и саму модель 
ResNet = VerificationModel(fe, frame_level, pooling, segment)

In [5]:
# загружаем веса
c = torch.load('ResNet.ckpt')
ResNet.load_state_dict(c)
# переводим на видеокарту и в eval-режим 
ResNet.to('cuda')
ResNet.eval()

VerificationModel(
  (fe): ExtractFbanks64()
  (frame_level): _ResNetModel(
    (conv1): Conv2d(1, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): ResNetBasicBlock(
        (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): ResNetBasicBlock(
        (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu):

In [6]:
from data import VerificationTestDataset
from torch.utils.data import DataLoader
from metrics.test import test_network 
# инициализируем даталоадер 
data_path = '/mnt/cs/voice/korenevskaya-a/VoicePersonification/'
test_dataset = VerificationTestDataset(scp_path, 
                                       '',
                                       protocol_path,
                                       16000)

test_loader  = DataLoader(test_dataset, batch_size=1, num_workers=5)
# считаем метрику ---Equal Error Rate в процентах 
test_network(test_loader, ResNet, protocol_path, 'cuda')


Start validation on /mnt/hs/dorado1/anikin/VoxCeleb1/protocols/VoxCeleb1-O...


100%|██████████| 4708/4708 [15:41<00:00,  5.00it/s]


{'EER': 0.5032775768535261, 'Thr': 0.9986307621002197}


### ResNetDEQ

In [9]:
# инициализируем модель
fe = ExtractFbanks64()
frame_level = ResNetModel.factory('deq_channels_96')
pooling = StatPoolLayer.by_string_mode('MV')
segment = MaxoutSegmentLevel(input_dim=1536, output_dim=512,enable_batch_norm=True)

ResNetDEQ = VerificationModel(fe, frame_level, pooling, segment)

In [10]:
ResNetDEQ 

VerificationModel(
  (fe): ExtractFbanks64()
  (frame_level): _ResNetModel(
    (conv1): Conv2d(1, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): DEQFixedPoint(
        (f): ResNetLayer(
          (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): DEQFixedPoint(
        (f): ResNetLayer(
          (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(96, eps=1e-05

In [11]:
# загружаем веса
c = torch.load('ResNetDEQ.ckpt')
ResNetDEQ.load_state_dict(c)
ResNetDEQ.to('cuda')
ResNetDEQ.eval()

VerificationModel(
  (fe): ExtractFbanks64()
  (frame_level): _ResNetModel(
    (conv1): Conv2d(1, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): DEQFixedPoint(
        (f): ResNetLayer(
          (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): DEQFixedPoint(
        (f): ResNetLayer(
          (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(96, eps=1e-05

In [13]:
# инициализируем модель 
data_path = '/mnt/cs/voice/korenevskaya-a/VoicePersonification/'
test_dataset = VerificationTestDataset(scp_path, 
                                       '',
                                       protocol_path,
                                       16000)

test_loader  = DataLoader(test_dataset, batch_size=1, num_workers=12)
# считаем Equal Error Rate на тестовом протоколе в процентах 
test_network(test_loader, ResNetDEQ, protocol_path, 'cuda')


Start validation on /mnt/hs/dorado1/anikin/VoxCeleb1/protocols/VoxCeleb1-O...


  0%|          | 0/4708 [00:00<?, ?it/s]

100%|██████████| 4708/4708 [1:38:52<00:00,  1.26s/it]  


{'EER': 0.49935113285820654, 'Thr': 0.997294545173645}
